In [204]:
import pandas as pd
import numpy as np

In [205]:
train = pd.read_csv("train_assignment.csv")
test = pd.read_csv("test_assignment.csv")

test_ids = test["id"]

train.head()

,id,marital_status_code,application_mode_code,application_order,course_code,attendance_shift_code,previous_qualification_code,previous_qualification_grade,nationality_code,mother_qualification_code,...,cu2_credited,cu2_enrolled,cu2_evaluations,cu2_approved,cu2_grade,cu2_without_evaluations,unemployment_rate,inflation_rate,gdp,Target
0,100001,1,1,2,9500,1,1,133.0,1,1,...,0,8,8,7,13.428571,0,12.7,3.7,-1.70,Graduate
1,100002,4,39,1,9853,1,12,133.1,1,34,...,0,6,7,0,0.000000,0,7.6,2.6,0.32,Dropout
2,100003,1,17,3,9773,1,1,133.0,1,38,...,0,6,11,4,12.500000,0,9.4,-0.8,-3.12,Enrolled
3,100004,1,1,2,171,1,1,133.1,1,37,...,0,0,0,0,0.000000,0,7.6,2.6,0.32,Dropout
4,100005,1,1,1,9238,1,1,124.0,1,1,...,0,6,6,6,11.500000,0,16.2,0.3,-0.92,Enrolled


In [206]:
X = train.drop(columns=["Target", "id"], errors='ignore')
y = train["Target"]

test = test.drop(columns=["id"], errors='ignore')

In [207]:
from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y)

In [208]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

In [209]:
from sklearn.impute import SimpleImputer

if len(num_cols) > 0:
    num_imputer = SimpleImputer(strategy='mean')
    X[num_cols] = num_imputer.fit_transform(X[num_cols])
    test[num_cols] = num_imputer.transform(test[num_cols])

if len(cat_cols) > 0:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])
    test[cat_cols] = cat_imputer.transform(test[cat_cols])

In [210]:
from sklearn.preprocessing import LabelEncoder

if len(cat_cols) > 0:
    for col in cat_cols:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])
        test[col] = le.transform(test[col])

In [ ]:
X["sum_features"] = X[num_cols].sum(axis=1)
test["sum_features"] = test[num_cols].sum(axis=1)

X["mean_features"] = X[num_cols].mean(axis=1)
test["mean_features"] = test[num_cols].mean(axis=1)

X["std_features"] = X[num_cols].std(axis=1)
test["std_features"] = test[num_cols].std(axis=1)

X["max_features"] = X[num_cols].max(axis=1)
test["max_features"] = test[num_cols].max(axis=1)

X["min_features"] = X[num_cols].min(axis=1)
test["min_features"] = test[num_cols].min(axis=1)

X["range_features"] = X["max_features"] - X["min_features"]
test["range_features"] = test["max_features"] - test["min_features"]

# interaction features 

X["sum_mean"] = X["sum_features"] * X["mean_features"]
test["sum_mean"] = test["sum_features"] * test["mean_features"]

X["range_std"] = X["range_features"] * X["std_features"]
test["range_std"] = test["range_features"] * test["std_features"]

In [212]:
X, test = X.align(test, join='left', axis=1, fill_value=0)

In [220]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')

print("CV Accuracy:", scores.mean())

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001550 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2690
[LightGBM] [Info] Number of data points in the train set: 61214, number of used features: 34
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001571 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2690
[LightGBM] [Info] Number of data points in the train set: 61214, number of used features: 34
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start 

In [ ]:
from lightgbm import LGBMClassifier

model = LGBMClassifier(
    n_estimators=2000,
    num_leaves=64,
    max_depth=-1,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_samples=20,
    class_weight='balanced',
    random_state=42
)

model.fit(X, y)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002337 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2827
[LightGBM] [Info] Number of data points in the train set: 76518, number of used features: 44
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612


,boosting_type,'gbdt'
,num_leaves,64
,max_depth,-1
,learning_rate,0.015
,n_estimators,2000
,subsample_for_bin,200000
,objective,None
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [ ]:
importances = model.feature_importances_

feat_imp = pd.DataFrame({
    'feature': X.columns,
    'importance': importances
}).sort_values(by='importance')


important_features = feat_imp[feat_imp["importance"] > 5]["feature"]

X = X[important_features]
test = test[important_features]

In [215]:
low_features = feat_imp.head(10)['feature'].values

X = X.drop(columns=low_features, errors='ignore')
test = test.drop(columns=low_features, errors='ignore')

In [216]:
X, test = X.align(test, join='left', axis=1, fill_value=0)

In [217]:
model.fit(X, y)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001785 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2708
[LightGBM] [Info] Number of data points in the train set: 76518, number of used features: 34
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612


,boosting_type,'gbdt'
,num_leaves,64
,max_depth,-1
,learning_rate,0.015
,n_estimators,2000
,subsample_for_bin,200000
,objective,None
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [218]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
rf.fit(X, y)

probs_lgb = model.predict_proba(test)
probs_rf = rf.predict_proba(test)

final_probs = (0.7 * probs_lgb + 0.3 * probs_rf)

preds = np.argmax(final_probs, axis=1)
preds = target_encoder.inverse_transform(preds)

In [219]:
submission = pd.DataFrame({
    "id": test_ids,
    "Target": preds
})

submission.to_csv("submission.csv", index=False)

print("Submission file created!")

Submission file created!
